# Encoding & Pipelines - Task 5: Final Mini-Project

**Week:** 5  
**Source:** `Encoding_Pipelines_PRACTICE.ipynb`

## 5. Final Mini-Project (Challenge)

Put everything together. No more step-by-step hand-holding.

**Your task:**
1. Build the best pipeline you can for predicting `Churn` on this
   dataset (you choose: Logistic Regression, Random Forest, or anything
   else from sklearn you're comfortable with).
2. Tune at least one hyperparameter with `GridSearchCV` or
   `cross_val_score`.
3. Report final **test set** accuracy AND a full `classification_report`.
4. Save your final trained pipeline to a file called
   `my_churn_pipeline.pkl` using `joblib`.
5. In a markdown cell below your code, write 3-4 sentences explaining:
   - Which encoding choices you made and why.
   - Whether `ColumnTransformer` + `Pipeline` made this easier than doing
     it manually, and specifically how.


### Prerequisites
Runs the setup and the earlier steps this task depends on (output hidden).

In [ ]:
%%capture
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

print("Setup complete.")

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df = df.drop(columns=['customerID'])

print("Shape:", df.shape)


df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values in TotalCharges:", df["TotalCharges"].isna().sum())

le = LabelEncoder()
df["Churn_encoded"] = le.fit_transform(df["Churn"])
print({cls: int(code) for cls, code in zip(le.classes_, le.transform(le.classes_))})
print(df[["Churn", "Churn_encoded"]].head())

X = df.drop(columns=["Churn", "Churn_encoded"])
y = df["Churn_encoded"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Why split first: frequency/target encoders learn statistics from the data; fitting them on the
# full dataset would leak information from the test set into training (data leakage).
print(X_train.shape, X_test.shape)

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
print("Numeric    :", numeric_features)
print("Categorical:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
preprocessor

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
log_reg_pipeline

log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

pipe_mp = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))
pipe_mp

pipe_mp.fit(X_train, y_train)
print("Pipeline     :", list(log_reg_pipeline.named_steps.keys()))
print("make_pipeline:", list(pipe_mp.named_steps.keys()))

### Solution

In [ ]:
import joblib

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])
param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [6, 10, None],
    "classifier__min_samples_leaf": [1, 5],
}
search = GridSearchCV(final_pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
search.fit(X_train, y_train)
best = search.best_estimator_
print("Best params:", search.best_params_)
print("CV accuracy:", round(search.best_score_, 4))

y_pred = best.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

joblib.dump(best, "my_churn_pipeline.pkl")
print("Saved my_churn_pipeline.pkl")

### Answer
I one-hot encoded the nominal columns (gender, InternetService, PaymentMethod, ...) because they have no natural order, and kept numeric columns scaled with median imputation, since logistic/forest models need clean numeric input. `ColumnTransformer` + `Pipeline` made this much easier than doing it by hand: all preprocessing is learned from the training split only (no leakage), the same object is reused unchanged on test data, and `GridSearchCV` can tune the model and the preprocessing together and the whole thing is saved in a single `.pkl` file.